# Introduction

Aujourd'hui la plupart des sites web utilisent du JavaScript pour rendre le site plus dynamique et agréable, mais aussi pour afficher la donnée. 

Pour ces sites, la première méthode abordée n'est pas efficace, car de nombreuses requêtes sont utilisées pour se connecter, afficher la donnée, naviguer, il faudrait prendre en compte toutes ces requêtes et les assembler pour simuler le comportement du site. 

**C'est le comportement de votre navigateur**, il compile toutes les interactions, les requêtes et génère du HTML permettant d'afficher l'interface graphique d'un site web.

Pour palier ce problème, on peut laisser le navigateur faire son travail et contrôler sa surcouche logicielle. Pour cela, on peut prendre le contrôle d'un navigateur comme Chrome, Edge, Firefox ou Safari depuis un script Python. 

Pour cela, nous allons utiliser un package Python `Selenium` https://selenium-python.readthedocs.io/ qui permet d'instancier un navigateur et de le contrôler. Sélénium est beaucoup utilisé pour générer des tests automatiques de sites web. 

On peut trouver les drivers des différents browser : 
- Chrome:	https://googlechromelabs.github.io/chrome-for-testing/#stable (bien télécharger le "chromedriver" et non pas chrome)
- Firefox:	https://github.com/mozilla/geckodriver/releases
- Safari:	https://webkit.org/blog/6900/webdriver-support-in-safari-10/

**Depuis les dernières versions de Selenium, il n'est plus néccessaires de télécharger les driver des browsers car Selenium les manage**

> **Version Docker (Mac Apple Silicon)** : Jupyter tourne dans un conteneur qui n'a ni Chrome ni écran.
> On ajoute donc un deuxième conteneur, `selenium` (image `selenium/standalone-chromium`), qui contient Chrome.
> Le notebook le pilote à distance avec `webdriver.Remote(...)` au lieu de `webdriver.Chrome()`.
> Pour **voir le navigateur bouger en direct**, ouvrir http://localhost:7900 dans son navigateur puis cliquer sur *Connect*.

In [1]:
from selenium import webdriver


In [2]:
# Version Docker : le navigateur tourne dans le conteneur "selenium" (voir docker-compose.yml).
# On s'y connecte à distance avec webdriver.Remote au lieu de webdriver.Chrome().
# Pour regarder le navigateur en direct : http://localhost:7900
SELENIUM_URL = "http://selenium:4444"

chrome = webdriver.Remote(command_executor=SELENIUM_URL, options=webdriver.ChromeOptions())

Vous devez voir apparaître une nouvelle instance de votre navigateur.
Chrome vous demande depuis peu de selectionner votre navigateur par défaut. Etant donné que notre but est de tout controller depuis notre code, on peut bypass cette demande en ajoutant des options au webdriver de Chrome

In [3]:
from selenium.webdriver.chrome.options import Options

chrome.quit()  # on ferme le premier navigateur avant d'en ouvrir un nouveau

chrome_options = Options()
chrome_options.add_argument("--disable-search-engine-choice-screen")

chrome = webdriver.Remote(command_executor=SELENIUM_URL, options=chrome_options)

Maintenant, vous avez le controlle du navigateur depuis Python. Si vous voulez accéder à une page :

In [4]:
chrome.get("https://books.toscrape.com/")

Allez maintenant voir votre fenêtre. Vous êtes sur un faux site de vente de livre. Ce site est spécialement concu pour être scrapé et comprendre les bases, mais en réalité scraper un site est un peu plus complexe, comme vous allez vous en rendre compte dans la section 2 sur Scrapy.

Vous pouvez maintenant vous déplacer dans la page. Pour cela, il va falloir inspecter le code source de la page.

Il existe de nombreuse méthode pour récupérer les éléments par exemple `find_element` qui permet de récupérer un élément grâce au texte affiché. Ici, si on veut récupérer cliquer sur la section `Travel` pour récupérer les livres de voyages

In [5]:
from selenium.webdriver.common.by import By
link = chrome.find_element(By.LINK_TEXT, "Travel")
type(link)

selenium.webdriver.remote.webelement.WebElement

Si on veut entrer dans le lien, il suffit d'appeler la méthode `click`.

In [6]:
link.click()

Vous êtes arrivé sur la page des livres de voyage. Maintenant nous allons récupérer toutes les livres. On remarque que tous les livres ont une class HTML nommée `product_pod`.

In [7]:
all_widgets = chrome.find_elements(By.CLASS_NAME, "product_pod")
type(all_widgets), len(all_widgets)

(list, 11)

On obtient une liste d'éléments décrits par cette class. Il y en a 11, un pour chaque livre. Pour récupérer le titre du premier livre: 

In [8]:
first_book = all_widgets[0]
first_book_title = first_book.find_element(By.CSS_SELECTOR, "h3")
first_book_title.text

"It's Only the Himalayas"

Si vous êtes fan de JavaScript vous pouvez même injecter du code JS dans le navigateur depuis Python pour exécuter des opérations complexes. 

In [9]:
js_script = """
const class_name = 'DSIA'
alert(`Hi from ${class_name}`)
"""

In [10]:
chrome.execute_script(js_script)

Une fenêtre d'alerte est maintenant ouverte dans le navigateur. Tant qu'elle est affichée, Selenium refuse toute autre action : on lit son texte puis on la ferme.

In [11]:
alert = chrome.switch_to.alert
print(alert.text)
alert.accept()  # équivalent du clic sur "OK"

Hi from DSIA


# Exercice

## Exercice 1

Extraire le prix des livres :

In [12]:
def extract_price(book_web_element):
    """Prix du livre, en float (ex : '£45.17' -> 45.17)."""
    price_text = book_web_element.find_element(By.CLASS_NAME, "price_color").text
    return float(price_text.replace("£", "").strip())

extract_price(all_widgets[0])

45.17

Extraire l'image des livres: 

In [13]:
def extract_image(book_web_element):
    """URL (absolue) de l'image de couverture du livre."""
    img = book_web_element.find_element(By.TAG_NAME, "img")
    return img.get_attribute("src")  # Selenium renvoie directement l'URL complète

extract_image(all_widgets[0])

'https://books.toscrape.com/media/cache/27/a5/27a53d0bb95bdd88288eaf66c9230d7e.jpg'

Extraire le nombre d'étoiles d'un livre: 

In [14]:
RATINGS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def extract_rating(book_web_element):
    """Nombre d'étoiles du livre (1 à 5).

    Dans le HTML, la note est écrite dans la classe : <p class="star-rating Three">.
    """
    stars = book_web_element.find_element(By.CLASS_NAME, "star-rating")
    for word in stars.get_attribute("class").split():
        if word in RATINGS:
            return RATINGS[word]
    return None

extract_rating(all_widgets[0])

2

Pour finir, on applique les trois fonctions à **tous les livres** de la page Travel et on range le résultat dans un tableau :

In [15]:
import pandas as pd

books = []
for book in all_widgets:
    books.append({
        "title": book.find_element(By.CSS_SELECTOR, "h3 a").get_attribute("title"),  # titre complet (le texte affiché est tronqué)
        "price": extract_price(book),
        "rating": extract_rating(book),
        "image": extract_image(book),
    })

df_books = pd.DataFrame(books)
df_books

,title,price,rating,image
0,It's Only the Himalayas,45.17,2,https://books.toscrape.com/media/cache/27/a5/2...
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,4,https://books.toscrape.com/media/cache/57/77/5...
2,See America: A Celebration of Our National Par...,48.87,3,https://books.toscrape.com/media/cache/9a/7e/9...
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,2,https://books.toscrape.com/media/cache/d5/bf/d...
4,Under the Tuscan Sun,37.33,3,https://books.toscrape.com/media/cache/98/c2/9...
5,A Summer In Europe,44.34,2,https://books.toscrape.com/media/cache/4e/15/4...
6,The Great Railway Bazaar,30.54,1,https://books.toscrape.com/media/cache/76/de/7...
7,A Year in Provence (Provence #1),56.88,4,https://books.toscrape.com/media/cache/db/46/d...
8,The Road to Little Dribbling: Adventures of an...,23.21,1,https://books.toscrape.com/media/cache/e0/4f/e...
9,Neither Here nor There: Travels in Europe,38.95,3,https://books.toscrape.com/media/cache/06/81/0...


In [16]:
chrome.quit()  # quit() ferme le navigateur ET libère la session dans le conteneur Selenium